## Managing Storage Buckets

# From Client to Bucket Management

Now that you've successfully set up your Google Cloud Storage client, it's time to put it to work. In this lesson, we'll focus exclusively on **bucket operations**—the fundamental containers that organize your cloud storage.

As you learned previously, buckets are globally unique containers that store your objects in specific locations. Let's explore how to create, configure, and manage these essential components.

---

## Creating Buckets in Different Locations

Each bucket requires a **globally unique name** across all Google Cloud projects. Once a name is taken, no other user in the world can use it until the original bucket is deleted.

### 1. Default Location

```python
from google.cloud import storage

storage_client = storage.Client()

# Create a new bucket in the default location (usually US)
bucket_name = "my-unique-bucket-12345"
bucket = storage_client.create_bucket(bucket_name)
print(f"Bucket created: {bucket.name}")

```

### 2. Specific Regions

Choosing a location is crucial for latency, compliance, and cost.

```python
# Create a bucket in Europe for GDPR compliance or local speed
eu_bucket = storage_client.create_bucket("my-eu-data-archive", location="europe-west1")

# Create a bucket on the US West Coast
us_bucket = storage_client.create_bucket("my-us-data-archive", location="us-west1")

```

---

## Listing and Inspecting Buckets

### Listing Buckets

```python
buckets = storage_client.list_buckets()
for bucket in buckets:
    print(f"- {bucket.name}")

```

### The Importance of `reload()`

When you use `storage_client.bucket("name")`, the SDK creates a "lazy" local object that doesn't actually call the API to fetch metadata. To see properties like `time_created`, you must call `reload()`.

```python
bucket = storage_client.bucket("my-existing-bucket")
print(bucket.time_created)  # Likely returns None

bucket.reload()  # Fetches the latest metadata from Google servers
print(bucket.time_created)  # Now shows actual creation time

```

---

## Configuration with `patch()`

Updating a bucket is a two-step process: modify the local object, then synchronize with the server using `patch()`.

```python
bucket = storage_client.get_bucket("my-archive-bucket")

# 1. Local modifications
bucket.storage_class = "COLDLINE"  # Optimized for long-term storage
bucket.labels = {"team": "data-science", "env": "prod"}

# 2. Server synchronization
bucket.patch()
print("Bucket properties updated on GCP servers.")

```

---

## Managing Bucket Lifecycle

### Safe Deletion

A bucket **must be empty** before it can be deleted. Attempting to delete a bucket containing files will result in a `Conflict` error.

```python
from google.api_core.exceptions import NotFound

def delete_bucket_safely(bucket_name):
    try:
        bucket = storage_client.get_bucket(bucket_name)
        
        # Check for existing objects
        blobs = list(bucket.list_blobs(max_results=1))
        if blobs:
            print("Abort: Bucket is not empty!")
            return
            
        bucket.delete()
        print("Bucket deleted successfully.")
    except NotFound:
        print("Bucket already gone.")

delete_bucket_safely("my-temp-bucket")

```

---

## Error Handling Best Practices

When creating buckets, the most common error is a `Conflict` (409), which occurs if the bucket name is already taken by you or **anyone else** in the world.

```python
from google.api_core.exceptions import Conflict, GoogleAPIError

try:
    storage_client.create_bucket("my-common-name")
except Conflict:
    print("This name is already taken. Try adding a random suffix.")
except GoogleAPIError as e:
    print(f"A service error occurred: {e.message}")

```

---

## Lesson Summary

* **Global Uniqueness**: Bucket names are shared globally.
* **Location Matters**: Placement impacts latency, price, and legal compliance.
* **Synchronization**: Use `reload()` to pull data from the cloud and `patch()` to push local changes back.
* **Deletion**: Buckets must be empty to be deleted; this action is permanent.

In the next unit, we'll dive inside the buckets to manage **Objects** (files), exploring how to upload, download, and move data.

## Google Cloud Storage Bucket Management Operations

Embark on a journey through Google Cloud Storage, where you're tasked with organizing your cosmic voyage memories and data across the galaxy. In this scenario, you will create three Cloud Storage buckets located in different regions to store photos from your intergalactic travels in 2024, widget data, and document archives from 2020. Witness firsthand how to manage these buckets within Google Cloud Platform: you'll see the listing of all your created buckets, perform a deletion operation on one, and then observe the updated list of buckets.

Simply click Run to see these Cloud Storage operations in action, showcasing the ease and power of managing cloud storage with Google Cloud Platform. No coding is required on your part for this task; enjoy watching your galactic storage system come to life!

Important Note: Running scripts can alter the filesystem's state or modify the resources in our GCP simulator. To revert to the initial state, you can use the reset button located in the top right corner. However, keep in mind that resetting will erase any code changes. To preserve your code during a reset, consider copying it to the clipboard.

```python
import os
from google.cloud import storage
from google.auth.credentials import AnonymousCredentials

storage_client = storage.Client(
    project=os.environ.get('PROJECT_ID'),
    credentials=AnonymousCredentials(),
    client_options={'api_endpoint': os.environ.get('STORAGE_HOST')}
)

storage_client.create_bucket('cosmo-galaxy-photos-2024')  # Creating bucket in the default location
storage_client.create_bucket('cosmo-widget-data', location='us-west1')  # Creating bucket in the us-west1 region
storage_client.create_bucket('cosmo-doc-archive-2020', location='europe-west1')  # Creating bucket in the europe-west1 region

print("Buckets before deletion:")
for bucket in storage_client.list_buckets():
    print(bucket.name)

storage_client.bucket('cosmo-doc-archive-2020').delete()

print("\nBuckets after deletion:")
for bucket in storage_client.list_buckets():
    print(bucket.name)
    
"""
Expected output:

Buckets before deletion:
cosmo-galaxy-photos-2024
cosmo-widget-data
cosmo-doc-archive-2020

Buckets after deletion:
cosmo-galaxy-photos-2024
cosmo-widget-data
"""

```

This simulation demonstrates the lifecycle of **Google Cloud Storage (GCS) Buckets**. Even though this script runs against a local emulator, it perfectly mirrors the logic used in production environments to manage global storage infrastructure.

### Key Takeaways from the Galactic Script

#### 1. Global vs. Regional Placement

Notice how the script places data strategically:

* **Default Location**: Used for `cosmo-galaxy-photos-2024`. This typically defaults to a multi-region location like `US`, providing high availability.
* **`us-west1` (Oregon)**: Used for `cosmo-widget-data`. You might choose this to minimize latency for users on the US West Coast.
* **`europe-west1` (Belgium)**: Used for `cosmo-doc-archive-2020`. Placing data in Europe is often a requirement for GDPR compliance or local data sovereignty laws.

#### 2. The Bucket Namespace

While the script creates these buckets easily, remember that in a real production environment, bucket names are **globally unique**. If someone else in the world had already created a bucket named `cosmo-widget-data`, your `create_bucket` call would fail with a `Conflict` error.

#### 3. Deletion is Permanent

The command `storage_client.bucket('cosmo-doc-archive-2020').delete()` is powerful and immediate.

* **Safety Check**: In this script, the bucket was empty, so it deleted successfully. If there were files (objects) inside, the command would have failed with an error, protecting you from accidental data loss.
* **Irreversibility**: Once a bucket is deleted, the name becomes available for anyone else to claim, and the data (if any was inside) is gone unless you have Versioning or Backup enabled.

---

### Understanding the Workflow

| Operation | SDK Method | Purpose |
| --- | --- | --- |
| **Creation** | `create_bucket()` | Reserves a unique global name and sets the physical location. |
| **Listing** | `list_buckets()` | Audits all storage containers currently billed to your project. |
| **Referencing** | `bucket('name')` | Creates a local pointer to a bucket without making a network call. |
| **Deletion** | `.delete()` | Removes the container from the Google Cloud network. |

By observing the output, you can see the "state" of your cloud environment changing in real-time. This foundational management is what allows developers to scale from a few personal photos to petabytes of enterprise data.

## Configure Google Cloud Storage Bucket Region

## Creating Regional GCP Storage Buckets for Space Mission Archives

## Delete Cloud Storage Bucket

## Final Mission: Complete Cloud Storage Bucket Management

## Final Mission: Complete Cloud Storage Bucket Management